<a href="https://colab.research.google.com/github/xyosho/Proyecto-Homologacion-ML/blob/Sprint-4/sprint4_explicabilidad_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4 — ¿Por qué el modelo toma sus decisiones? (Machine Learning explicable)
**Pedido del docente (Semana 4):** un análisis que responda *"¿Por qué el modelo toma sus decisiones?"* con SHAP o
LIME. Por ejemplo, mostrar que el modelo le da más peso al ICFES que a la edad, con una explicación en español sencillo
de por qué eso tiene sentido para la Universidad.

| Sección | Contenido |
|---|---|
| 1 | Qué es SHAP y cómo se agrupan las variables en conceptos de negocio |
| 2 | Importancia global (modelo elegido vs. XGBoost) |
| 3 | Cómo cambia cada factor: descuento, distancia, Saber 11 y canal |
| 4 | Tres inscritos de la campaña 2027-1: SHAP vs. LIME |
| 5 | Chequeo independiente: importancia por permutación |
| 6 | Coherencia con las reglas de negocio y equidad por género |
| 7 | ¿El modelo encontró las causas reales? (comparación con el generador) |
| 8 | Explicaciones para el tablero (bloque 4) |

## Antes de empezar (Google Colab)
Las credenciales **no** se escriben en ninguna celda; se guardan en los **Secretos** de Colab:

1. Abra el panel **Secretos** (icono de llave, a la izquierda).
2. Cree `SUPABASE_URL` (la URL del proyecto, `https://….supabase.co`) y `SUPABASE_KEY` (la clave **publicable**,
   `sb_publishable_…`; nunca la secreta).
3. Active **Acceso del notebook** en los dos secretos.
4. Ejecute todo: *Entorno de ejecución → Ejecutar todas*. Si aparece un aviso para dar acceso a los secretos, acéptelo.

Las celdas que empiezan con `%%writefile` crean en Colab una copia exacta del código del repositorio (carpeta `src/`).

In [ ]:
!pip -q install supabase==2.33.0 xgboost==3.4.1 shap==0.54.1 lime==0.2.0.1

In [ ]:
%%writefile comun.py
"""Versión para Google Colab de src/comun.py: la configuración del proyecto (config/reglas_negocio.yaml)
viene incrustada porque Colab no tiene el repositorio. Generado por src/s06_construir_cuadernos.py."""
from pathlib import Path

RAIZ = Path.cwd()
DATA_PROCESSED = RAIZ / "data" / "processed"

_CONFIG = {'campus': {'nombre': 'Campus Unicervantes',
            'direccion': 'Calle 209 No. 104-15, vía Arrayanes, Bogotá',
            'lat': 4.790438,
            'lon': -74.066288,
            'fuente': 'Dirección: Acuerdo 002/2025 (membrete). Coordenada exacta de la placa Calle 209 # '
                      '104-15 (CL 209 104 15, lote 009140001001) en Placa Domiciliaria de Catastro Bogotá '
                      '(UAECD), consultada 23-oct-2026. Antes se usaba una coordenada aproximada de '
                      'OpenStreetMap (4.7895, -74.0660), a unos 100 m.'},
 'tarifas': {'AD': {2024: {'valor': 4419000,
                           'fuente': 'Precio de matrícula según recibos de pago: orden de pago Pregrado '
                                     "2024-2, concepto 'MAT ING ANALITICA DE DATOS NUEVOS'."},
                    2025: {'valor': 4658000,
                           'fuente': 'Precio de matrícula según recibos de pago 2025-1 y 2025-2: $3.260.600 '
                                     'con beca del 30% => 3.260.600 / 0,70; coincide con la versión 2025 de '
                                     'la página del programa.'},
                    2026: {'valor': 4914000,
                           'fuente': 'Acuerdo 002 del 11 de noviembre de 2025 del Consejo Directivo, art. 2 '
                                     '(valores de matrícula 2026).'},
                    2027: {'valor': 4914000,
                           'fuente': 'supuesto: S-01 (se mantiene el valor vigente hasta que se publique el '
                                     'acuerdo 2027).'}},
             'SSIA': {2026: {'valor': 4950000,
                             'fuente': 'Oferta académica oficial '
                                       'unicervantes.edu.co/admision/oferta-academica (consultada '
                                       '19-oct-2026).'},
                      2027: {'valor': 4950000, 'fuente': 'supuesto: S-01.'}}},
 'becas': {'SABER_11': {'nombre': 'Beca Saber 11',
                        'umbral_icfes': 310,
                        'pct_primer_periodo': 0.35,
                        'pct_siguientes': 0.3,
                        'promedio_minimo': 4.0,
                        'fuente': 'Admisiones (corrección del 31-oct-2026): 35% el 1er semestre y 30% desde '
                                  'el 2º; coincide con los recibos (2024-2: 35%; 2025 y 2026: 30%). Página '
                                  "oficial: 'Puntaje igual o superior a 310 ... 30% para toda la carrera por "
                                  "promedio académico'."},
           'MI_VOCACION': {'nombre': 'Beca Cervantes #MiVocación',
                           'pct_primer_periodo': 0.3,
                           'pct_siguientes': 0.2,
                           'promedio_minimo': 4.0,
                           'fuente': "Admisiones ('30% para el primer periodo académico y en adelante según "
                                     "la tabla relacionada'; se mantiene con promedio acumulado >= 4.0 y se "
                                     "pierde por sanciones); landing del programa ('30% 1er semestre y 20% "
                                     "el resto de la carrera')."},
           'NINGUNA': {'nombre': 'Sin beca',
                       'pct_primer_periodo': 0.0,
                       'pct_siguientes': 0.0,
                       'promedio_minimo': None},
           'acumulables': False},
 'particion': {'entrenamiento': ['2024-1', '2024-2', '2025-1', '2025-2'],
               'validacion': ['2026-1'],
               'prueba': ['2026-2'],
               'campana': ['2027-1']},
 'caracteristicas': {'bandas_icfes': [{'desde': 0, 'hasta': 250, 'segmento': 'BAJO'},
                                      {'desde': 250, 'hasta': 310, 'segmento': 'MEDIO'},
                                      {'desde': 310, 'hasta': 350, 'segmento': 'ALTO'},
                                      {'desde': 350, 'hasta': 501, 'segmento': 'SUPERIOR'}],
                     'zonas_distancia_km': {'CERCA': 10, 'MEDIA': 20},
                     'canal_agrupado': {'EVENTO': ['EXPOESTUDIANTES', 'CAMPUS_DAY'],
                                        'RELACIONAL': ['REFERIDO', 'INMERSION', 'COLEGIO_CONVENIO'],
                                        'DIGITAL': ['SITIO_WEB', 'REDES_SOCIALES', 'LLAMADA_WHATSAPP']},
                     'estrato_grupo': {1: 'BAJO', 2: 'BAJO', 3: 'MEDIO', 4: 'ALTO', 5: 'ALTO', 6: 'ALTO'},
                     'kmeans': {'k_min': 3,
                                'k_max': 7,
                                'tolerancia_silueta': 0.01,
                                'semilla': 20261029,
                                'variables': ['estrato',
                                              'log_distancia',
                                              'punt_global',
                                              'promedio_acumulado_colegio',
                                              'edad_al_inscribirse',
                                              'colegio_privado']}},
 'modelado': {'semilla': 20261105,
              'umbral_clasificacion': 0.5,
              'segmentos_propension': {'ALTA': 0.6, 'BAJA': 0.35},
              'sprint2_logistica': {'C': 1.0},
              'grillas': {'logistica': {'C': [0.01, 0.1, 1.0, 10.0]},
                          'random_forest': {'max_depth': [4, 6, 8, None],
                                            'min_samples_leaf': [5, 20],
                                            'max_features': ['sqrt', 0.5]},
                          'xgboost': {'n_estimators': [200, 400],
                                      'max_depth': [2, 3, 4],
                                      'learning_rate': [0.03, 0.1],
                                      'min_child_weight': [5]}}},
 'impacto': {'seguimiento': {'capacidad_pct': 0.3, 'efecto_maximo': 0.1, 'costo_por_inscrito': 30000},
             'expo': {'inversion_evento': 22000000}},
 'escala_desempeno_bachillerato': {'SUPERIOR': 4.6, 'ALTO': 4.0, 'BASICO': 3.0},
 'fondos_educacion_municipal': {'FOES_CHIA': {'ubicacion_id': 25175,
                                              'nombre': 'FOES - Fondo para el Fomento de la Educación '
                                                        'Superior de Chía',
                                              'norma': 'Acuerdo 172 de 2020; Resolución 2114 de 2021',
                                              'convenio_unicervantes': 'VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Chía (certificado de la JAC), '
                                                            'registro en el SISBÉN de Chía, certificado de '
                                                            'notas, Saber 11 y recibo de matrícula',
                                              'renovacion': 'Cada semestre (abril a octubre) según el '
                                                            'promedio del semestre anterior fijado en el '
                                                            'convenio',
                                              'fuente': 'Alcaldía de Chía (2022). Conozca el programa FOES: '
                                                        'Unicervantes en convenio (n.º 15); apoyo educativo '
                                                        'desde $1.000.000 en carreras presenciales.',
                                              'url': 'https://www.chia-cundinamarca.gov.co/2022/educacion/PROGRAMA%20FOES%20ACTUALIZADO%2022-09-2022.pdf'},
                                'FOES_SOPO': {'ubicacion_id': 25758,
                                              'nombre': 'FOES - Fondo Especial para el Fomento de la '
                                                        'Educación de Sopó',
                                              'norma': 'No publicada en la página consultada',
                                              'convenio_unicervantes': 'NO_VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Sopó, admitido o matriculado en un '
                                                            'programa de educación superior',
                                              'renovacion': 'No publicada',
                                              'fuente': 'Alcaldía de Sopó (2022, actualizado 2026). FOES: '
                                                        'apoyo de matrícula, transporte y sostenimiento. '
                                                        'Monto: supuesto S-31.',
                                              'url': 'https://www.sopo-cundinamarca.gov.co/educacion/publicaciones/290/foes-fondo-especial-para-el-fomento-de-la-educacion/'},
                                'FOEST_TOCANCIPA': {'ubicacion_id': 25817,
                                                    'nombre': 'FOEST - Fondo Especial de Educación Superior '
                                                              'de Tocancipá',
                                                    'norma': 'Aprobado por el Concejo Municipal (acuerdo no '
                                                             'consultado)',
                                                    'convenio_unicervantes': 'NO_VERIFICADO',
                                                    'apoyo_semestral_cop': 1000000,
                                                    'requisitos': 'Residente de Tocancipá; becas para '
                                                                  'mejores puntajes Saber 11, artistas y '
                                                                  'deportistas; descuentos por convenio',
                                                    'renovacion': 'No publicada',
                                                    'fuente': 'Alcaldía de Tocancipá (2017). $1.750 millones '
                                                              'para la educación superior. Monto por '
                                                              'estudiante: supuesto S-31.',
                                                    'url': 'https://www.tocancipa-cundinamarca.gov.co/sala-prensa/noticias-1/1-750-millones-para-educacion-superior-tocancipa'}},
 'inversion_captacion': {'por_evento': {'EXPOESTUDIANTES': 22000000, 'CAMPUS_DAY': 6000000},
                         'por_periodo': {'SITIO_WEB': {'1': 9000000, '2': 6000000},
                                         'REDES_SOCIALES': {'1': 18000000, '2': 10000000},
                                         'LLAMADA_WHATSAPP': {'1': 8000000, '2': 6000000},
                                         'COLEGIO_CONVENIO': {'1': 7000000, '2': 4000000},
                                         'INMERSION': {'1': 5000000, '2': 5000000},
                                         'REFERIDO': {'1': 3000000, '2': 3000000}},
                         'ventana_campana': {'1': ['03-01', '01-25'], '2': ['01-10', '07-25']}},
 'generador': {'matricula': {'tasa_objetivo': 0.45,
                             'beta_descuento': 3.0,
                             'beta_estrato': {1: -0.5, 2: -0.15, 3: 0.0, 4: 0.1, 5: 0.0, 6: -0.3},
                             'beta_log_distancia': -0.35,
                             'beta_icfes_cuadratico': -0.25,
                             'beta_canal': {'EXPOESTUDIANTES': -0.3,
                                            'SITIO_WEB': 0.0,
                                            'REDES_SOCIALES': -0.1,
                                            'REFERIDO': 0.5,
                                            'CAMPUS_DAY': 0.3,
                                            'LLAMADA_WHATSAPP': -0.2,
                                            'INMERSION': 0.8,
                                            'COLEGIO_CONVENIO': 0.3},
                             'beta_entrevista': 0.6,
                             'beta_programa_ssia': 0.1,
                             'beta_periodo_2': -0.1,
                             'beta_foes': 0.5}}}


def cargar_config() -> dict:
    return _CONFIG


In [ ]:
%%writefile conexion_supabase.py
"""Conexión SEGURA de Python con Supabase para leer la tabla plana dataset_matricula.

Las credenciales NUNCA se escriben en el código ni se imprimen:
  - En el computador: archivo .env en la raíz del proyecto, excluido por .gitignore (plantilla: .env.example).
    Se puede crear con:  python src/s00_configurar_credenciales.py
  - En Google Colab: panel "Secretos" (icono de llave) con SUPABASE_URL y SUPABASE_KEY, con acceso al cuaderno.

SUPABASE_KEY es la clave PUBLICABLE del proyecto. En la base, la seguridad a nivel de fila (RLS) solo le permite
LEER dataset_matricula (sin datos personales); las otras 25 tablas están cerradas y nadie puede escribir con ella.
Así, aunque la clave se filtrara, solo se podría leer el dataset anonimizado.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

TABLA = "dataset_matricula"
VARIABLES = ("SUPABASE_URL", "SUPABASE_KEY")
TAMANO_PAGINA = 1000          # la API de Supabase entrega como máximo 1.000 filas por petición
TIEMPO_LIMITE_S = 30          # si la red no responde, se desiste en vez de quedarse esperando


def en_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def obtener_credenciales() -> tuple[str, str]:
    """Devuelve (url, clave) desde los Secretos de Colab o desde .env / variables de entorno.

    En Colab, la primera vez que un cuaderno lee un secreto aparece un aviso para darle acceso: hay que aceptarlo
    (mientras no se acepte, la celda espera). Los errores explican qué falta, nunca muestran los valores.
    """
    problemas = []
    if en_colab():
        from google.colab import userdata
        valores = {}
        for v in VARIABLES:
            try:
                valores[v] = (userdata.get(v) or "").strip()
            except Exception as e:                  # SecretNotFoundError / NotebookAccessError
                valores[v] = ""
                motivo = ("el cuaderno no tiene acceso: active 'Acceso del notebook'" if "Access" in type(e).__name__
                          else "no existe un secreto con ese nombre exacto")
                problemas.append(f"{v}: {motivo}")
        donde = "los Secretos de Colab (icono de llave, con 'Acceso del notebook' activado)"
    else:
        from dotenv import load_dotenv
        load_dotenv(Path(__file__).resolve().parents[1] / ".env")
        valores = {v: os.environ.get(v, "").strip() for v in VARIABLES}
        donde = "el archivo .env (python src/s00_configurar_credenciales.py o copia de .env.example)"
    problemas += [f"{v}: está vacío" for v, x in valores.items() if not x and not any(p.startswith(v) for p in problemas)]
    url, clave = valores["SUPABASE_URL"], valores["SUPABASE_KEY"]
    if url and not (url.startswith("https://") and url.rstrip("/").endswith(".supabase.co")):
        problemas.append("SUPABASE_URL: no es la URL del proyecto (debe ser https://<proyecto>.supabase.co; "
                         "no la cadena postgresql:// del pooler)")
    if clave.lower().startswith(("postgres://", "postgresql://")):
        problemas.append("SUPABASE_KEY: es la cadena del pooler; aquí va la clave PUBLICABLE (sb_publishable_...)")
    elif clave.startswith("sb_secret_"):
        problemas.append("SUPABASE_KEY: es la clave SECRETA; use la PUBLICABLE (sb_publishable_...)")
    if problemas:
        raise RuntimeError(f"Credenciales incompletas en {donde}: " + "; ".join(problemas) + ".")
    return url, clave


def crear_cliente(url: str | None = None, clave: str | None = None):
    """Cliente de la API de Supabase con tiempo límite. La clave se usa en memoria y no se muestra."""
    from supabase import ClientOptions, create_client
    if url is None or clave is None:
        url, clave = obtener_credenciales()
    return create_client(url, clave, options=ClientOptions(postgrest_client_timeout=TIEMPO_LIMITE_S))


def leer_dataset_matricula(cliente, tabla: str = TABLA) -> pd.DataFrame:
    """Lee la tabla completa en páginas de 1.000 filas (límite de la API), ordenada por inscripcion_id."""
    filas, inicio = [], 0
    while True:
        r = (cliente.table(tabla).select("*").order("inscripcion_id")
             .range(inicio, inicio + TAMANO_PAGINA - 1).execute())
        filas += r.data
        if len(r.data) < TAMANO_PAGINA:
            break
        inicio += TAMANO_PAGINA
    return pd.DataFrame(filas)


def probar_solo_lectura(cliente, tabla: str = TABLA) -> bool:
    """Intenta modificar la tabla con la clave publicable. Devuelve True si la base lo RECHAZA por permisos
    (código 42501 de PostgreSQL: permiso denegado). La fila -1 no existe, así que nada cambiaría de todos modos."""
    try:
        cliente.table(tabla).update({"punt_global": 0}).eq("inscripcion_id", -1).execute()
    except Exception as e:                           # APIError de la API de Supabase
        return getattr(e, "code", None) == "42501"
    return False


In [ ]:
%%writefile datos.py
"""Carga de las tablas planas para los sprints 2 a 6.

Fuente preferida: Supabase con la API de solo lectura (credenciales en .env o en los Secretos de Colab, ver
conexion_supabase.py). Si no hay credenciales, se usan los CSV que exporta el pipeline (data/processed), que son
idénticos (las pruebas lo comprueban). La función siempre dice de dónde salieron los datos.
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

TABLAS = ("dataset_matricula", "dataset_entrevista", "resultado_campana_2027_1")   # el último solo en el Sprint 5
ENTEROS_CON_VACIOS = ("matriculado", "puntaje_vocacional", "matriculado_real", "entrevista_interes_programa",
                      "entrevista_proyecto_vida", "entrevista_interes_universidad", "entrevista_conciencia_social",
                      "entrevista_expresion")


def preparar_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """La API devuelve JSON: se fijan los tipos para que local y Colab den exactamente lo mismo."""
    if df.empty:                                   # tabla sin filas: no hay tipos que fijar
        return df
    df = df.copy()
    for c in ENTEROS_CON_VACIOS:
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype("Int64")
    for c in ("distancia_campus_km", "distancia_vial_km", "colegio_promedio_saber11", "promedio_acumulado_colegio",
              "promedio_matematicas_colegio", "pct_descuento_inicial"):
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype(float)
    if "asesor_id" in df:
        df["asesor_id"] = df["asesor_id"].astype(int)
    return df.sort_values("inscripcion_id").reset_index(drop=True)


def _csv_local(tabla: str) -> Path:
    from comun import DATA_PROCESSED
    if tabla == "resultado_campana_2027_1":       # "sobre sellado" desde la Semana 1 (data/processed/oculto)
        return DATA_PROCESSED / "oculto" / "resultado_2027_1_sellado.parquet"
    return DATA_PROCESSED / f"{tabla}.csv"


def cargar(tabla: str, fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """Devuelve (datos, origen). fuente: 'supabase', 'local' o 'auto' (Supabase si hay credenciales)."""
    if tabla not in TABLAS:
        raise ValueError(f"Tabla desconocida: {tabla}")
    if fuente in ("auto", "supabase"):
        try:
            from conexion_supabase import crear_cliente, leer_dataset_matricula
            df = leer_dataset_matricula(crear_cliente(), tabla=tabla)
            return preparar_tipos(df), "Supabase (API de solo lectura)"
        except Exception as error:
            if fuente == "supabase":
                raise
            aviso = type(error).__name__
    ruta = _csv_local(tabla)
    df = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta)
    origen = "CSV local exportado por el pipeline" + (f" (sin conexión a Supabase: {aviso})" if fuente == "auto" else "")
    return preparar_tipos(df), origen


def cargar_inscritos(fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """dataset_matricula con la rúbrica de la entrevista (dataset_entrevista) unida por inscripcion_id."""
    inscritos, origen = cargar("dataset_matricula", fuente)
    entrevista, _ = cargar("dataset_entrevista", fuente)
    if len(entrevista) != len(inscritos):     # p. ej. la tabla existe en Supabase pero aún no se cargó (src/s05)
        if fuente == "supabase":
            raise RuntimeError(f"dataset_entrevista tiene {len(entrevista)} filas y dataset_matricula {len(inscritos)}: "
                               "falta cargar la entrevista en Supabase (python src/s05_cargar_supabase.py).")
        entrevista, _ = cargar("dataset_entrevista", "local")
    return inscritos.merge(entrevista, on="inscripcion_id", how="left", validate="one_to_one"), origen


In [ ]:
%%writefile caracteristicas.py
"""SPRINT 2 — Ingeniería de características: columnas nuevas y útiles sobre la tabla plana dataset_matricula.

Regla de oro contra la FUGA DE DATOS: todo lo que se APRENDE de los datos (las medias para estandarizar y los grupos
de k-means) se ajusta SOLO con el conjunto de ENTRENAMIENTO
(2024-1 a 2025-2) y después se aplica igual a validación, prueba, campaña y a cualquier aspirante nuevo (Sprint 6).
Si se ajustara con todos los datos, el modelo "vería" información del futuro y sus métricas serían optimistas.

Funciones principales:
  particion_temporal(periodo)                      -> ENTRENAMIENTO / VALIDACION / PRUEBA / CAMPANA
  ajustar_referencias(inscritos)                   -> lo aprendido del entrenamiento (un diccionario)
  construir_caracteristicas(inscritos, referencias) -> la misma tabla con las columnas nuevas
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

NUEVAS = [
    "particion", "aplica_beca_saber11", "aplica_beca_mi_vocacion", "indice_vocacional",
    "segmento_rendimiento_icfes", "brecha_colegio_vs_icfes",
    "ahorro_beca_primer_semestre", "log_distancia", "zona_distancia", "rezago_saber11",
    "canal_agrupado", "estrato_grupo", "colegio_privado", "segmento_mercado",
]

# Las 5 secciones de la entrevista de admisión calificadas de 1 a 5 (dataset_entrevista, supuesto S-46)
DIMENSIONES_ENTREVISTA = ["entrevista_interes_programa", "entrevista_proyecto_vida", "entrevista_interes_universidad",
                          "entrevista_conciencia_social", "entrevista_expresion"]

# Descripción de cada columna nueva (diccionario del Sprint 2)
DICCIONARIO = {
    "particion": "Conjunto temporal: ENTRENAMIENTO (2024-1 a 2025-2), VALIDACION (2026-1), PRUEBA (2026-2), CAMPANA (2027-1)",
    "aplica_beca_saber11": "Sí/No: puntaje global Saber 11 ≥ 310 (requisito de la Beca Saber 11)",
    "aplica_beca_mi_vocacion": "Sí/No: presentó entrevista y su puntaje vocacional es ≥ 3 (requisito de Mi Vocación)",
    "indice_vocacional": "Índice de 0 a 100 de la entrevista de admisión: 25 × (promedio de la rúbrica 1-5 de sus 5 secciones − 1); vacío si no hubo entrevista",
    "segmento_rendimiento_icfes": "BAJO (< 250), MEDIO (250-309), ALTO (310-349: ya aplica a beca) o SUPERIOR (≥ 350)",
    "brecha_colegio_vs_icfes": "z(promedio del colegio) − z(Saber 11): positivo = rinde más en el colegio que en el examen",
    "ahorro_beca_primer_semestre": "Cuánto le ahorra la beca en el 1er semestre (tarifa plena − precio neto), en COP",
    "log_distancia": "Logaritmo natural de la distancia por vías desde la dirección hasta el campus (S-44; reduce la cola larga)",
    "zona_distancia": "Distancia por vías desde la dirección: CERCA (≤ 10 km), MEDIA (10-20 km) o LEJOS (> 20 km)",
    "rezago_saber11": "INMEDIATO (presentó el Saber 11 el año anterior), UN_ANIO o DOS_O_MAS",
    "canal_agrupado": "EVENTO (Expo, Campus Day), RELACIONAL (referido, inmersión, convenio) o DIGITAL (web, redes, llamada)",
    "estrato_grupo": "BAJO (1-2), MEDIO (3) o ALTO (4-6)",
    "colegio_privado": "1 si el colegio es privado",
    "segmento_mercado": "Perfil del aspirante según k-means (segmentación de mercado), nombrado por sus rasgos principales",
}

ETIQUETAS_KMEANS = {   # (si el centro está por encima, si está por debajo) de la media del entrenamiento
    "estrato": ("estrato más alto", "estrato más bajo"),
    "log_distancia": ("viven lejos del campus", "viven cerca del campus"),
    "punt_global": ("buen puntaje ICFES", "puntaje ICFES bajo"),
    "promedio_acumulado_colegio": ("buenas notas en el colegio", "notas bajas en el colegio"),
    "edad_al_inscribirse": ("aspirantes mayores", "aspirantes más jóvenes"),
    "colegio_privado": ("colegio privado", "colegio público"),
    "colegio_promedio_saber11": ("colegio de alto nivel", "colegio de nivel bajo"),
}

# Nombres en español sencillo para tablas y gráficas (los nombres técnicos de las columnas no cambian)
NOMBRES_LEGIBLES = {
    "estrato": "Estrato promedio",
    "log_distancia": "Distancia por vías (km)",
    "distancia_vial_km": "Distancia por vías (km)",
    "punt_global": "Puntaje ICFES",
    "promedio_acumulado_colegio": "Notas del colegio (1 a 5)",
    "edad_al_inscribirse": "Edad (años)",
    "colegio_privado": "% de colegio privado",
    "colegio_promedio_saber11": "Nivel ICFES del colegio",
    "segmento_mercado": "Segmento de mercado",
    "segmento_rendimiento_icfes": "Puntaje ICFES",
    "aplica_beca_mi_vocacion": "¿Aplica a la beca Mi Vocación?",
    "aplica_beca_saber11": "¿Aplica a la beca Saber 11?",
    "canal_agrupado": "Canal por el que llegó",
    "zona_distancia": "Distancia al campus",
    "estrato_grupo": "Estrato",
    "rezago_saber11": "Tiempo desde el ICFES",
    "indice_vocacional": "Índice vocacional de la entrevista (0 a 100)",
}

# El ICFES llama "oficial" al colegio público. Los datos y los modelos conservan el código OFICIAL; en todo texto que
# se muestra (explicaciones, gráficas, script de predicción) se dice "público".
NOMBRE_TIPO_COLEGIO = {"OFICIAL": "público", "PRIVADO": "privado", "TECNICO": "técnico"}


def tipo_colegio_codigo(valor: str) -> str:
    """Código de datos del tipo de colegio: acepta PUBLICO (o PÚBLICO) como sinónimo de OFICIAL."""
    v = str(valor).strip().upper().replace("Ú", "U").replace("É", "E")
    return "OFICIAL" if v == "PUBLICO" else v


def etiquetas_categorias(cfg: dict | None = None) -> dict:
    """Texto claro para los códigos de las columnas categóricas nuevas (p. ej. CERCA → "Cerca (hasta 10 km)")."""
    c = (cfg or configuracion())["caracteristicas"]
    z = c["zonas_distancia_km"]
    icfes = {b["segmento"]: (f"{b['segmento'].capitalize()} (menos de {b['hasta']})" if b["desde"] == 0 else
                             f"{b['segmento'].capitalize()} ({b['desde']} o más)" if b["hasta"] > 500 else
                             f"{b['segmento'].capitalize()} ({b['desde']} a {b['hasta'] - 1})") for b in c["bandas_icfes"]}
    return {
        "segmento_rendimiento_icfes": icfes,
        "zona_distancia": {"CERCA": f"Cerca (hasta {z['CERCA']} km)", "MEDIA": f"Media ({z['CERCA']} a {z['MEDIA']} km)",
                           "LEJOS": f"Lejos (más de {z['MEDIA']} km)"},
        "canal_agrupado": {"EVENTO": "Eventos (Expo, Campus Day)", "RELACIONAL": "Relacional (referido, convenio…)",
                           "DIGITAL": "Digital (web, redes, llamada)"},
        "estrato_grupo": {"BAJO": "Estratos 1 y 2", "MEDIO": "Estrato 3", "ALTO": "Estratos 4 a 6"},
        "rezago_saber11": {"INMEDIATO": "Sin pausa tras el colegio", "UN_ANIO": "1 año de pausa", "DOS_O_MAS": "2 años o más de pausa"},
        "aplica_beca_mi_vocacion": {"Sí": "Sí aplica", "No": "No aplica"},
        "aplica_beca_saber11": {"Sí": "Sí aplica", "No": "No aplica"},
    }


def tabla_segmentos(d: pd.DataFrame) -> pd.DataFrame:
    """Resumen de cada segmento de mercado con valores reales (no estandarizados), listo para leer o presentar."""
    m = d.assign(matriculado=d["matriculado"].astype(float))
    def resumen(g: pd.DataFrame) -> dict:
        return {"Inscritos": len(g), "% del total": len(g) / len(d) * 100,
                "% que se matriculó": g["matriculado"].mean() * 100, "Puntaje ICFES": g["punt_global"].mean(),
                "Notas del colegio (1 a 5)": g["promedio_acumulado_colegio"].mean(),
                "Distancia por vías (km)": g["distancia_vial_km"].mean(), "Estrato promedio": g["estrato"].mean(),
                "% de colegio público": (1 - g["colegio_privado"].mean()) * 100, "Edad (años)": g["edad_al_inscribirse"].mean()}
    t = pd.DataFrame({s: resumen(g) for s, g in m.groupby("segmento_mercado")}).T
    t.loc["Todos los inscritos"] = resumen(m)
    t.index.name = "Segmento de mercado"
    return t.round({"Inscritos": 0, "% del total": 1, "% que se matriculó": 1, "Puntaje ICFES": 0,
                    "Notas del colegio (1 a 5)": 2, "Distancia por vías (km)": 1, "Estrato promedio": 1,
                    "% de colegio público": 0, "Edad (años)": 1}).astype({"Inscritos": int})


def configuracion() -> dict:
    """Parámetros del proyecto: config/reglas_negocio.yaml en el computador; en Colab, la copia del cuaderno."""
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


# ----------------------------------------------------------------------------- reglas (no se aprenden de datos)
def particion_temporal(periodo: pd.Series, cfg: dict | None = None) -> pd.Series:
    p = (cfg or configuracion())["particion"]
    mapa = {per: nombre.upper() for nombre in ("entrenamiento", "validacion", "prueba", "campana") for per in p[nombre]}
    return periodo.map(mapa)


def segmento_icfes(punt_global: pd.Series, cfg: dict | None = None) -> pd.Series:
    bandas = (cfg or configuracion())["caracteristicas"]["bandas_icfes"]
    cortes = [bandas[0]["desde"]] + [b["hasta"] for b in bandas]
    return pd.cut(punt_global, bins=cortes, right=False, labels=[b["segmento"] for b in bandas]).astype(str)


def indice_vocacional(df: pd.DataFrame) -> pd.Series:
    """Entrevista cualitativa → número: 25 × (promedio de las 5 secciones calificadas de 1 a 5 − 1), de 0 a 100.
    Vacío si no hubo entrevista. Es una regla fija (no se aprende de los datos), así que no hay riesgo de fuga."""
    return (25 * (df[DIMENSIONES_ENTREVISTA].astype(float).mean(axis=1) - 1)).round(1)


def _reglas(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    c = cfg["caracteristicas"]
    d = df.copy()
    d["particion"] = particion_temporal(d["periodo"], cfg)
    d["aplica_beca_saber11"] = np.where(d["punt_global"] >= cfg["becas"]["SABER_11"]["umbral_icfes"], "Sí", "No")
    d["aplica_beca_mi_vocacion"] = np.where(d["cumple_req_mi_vocacion"] == 1, "Sí", "No")
    d["indice_vocacional"] = indice_vocacional(d)
    d["segmento_rendimiento_icfes"] = segmento_icfes(d["punt_global"], cfg)
    # El precio que se usa es el neto del 1er semestre (valor_neto_primer_semestre, ya en dataset_matricula)
    d["ahorro_beca_primer_semestre"] = d["valor_semestre_bruto"] - d["valor_neto_primer_semestre"]
    vial = d["distancia_vial_km"].astype(float)                 # desde la dirección del formulario (S-44)
    d["log_distancia"] = np.log(np.maximum(vial, 0.5))           # piso de 0,5 km: un vecino del campus no da −∞
    z = c["zonas_distancia_km"]
    d["zona_distancia"] = np.select([vial <= z["CERCA"], vial <= z["MEDIA"]],
                                    ["CERCA", "MEDIA"], "LEJOS")
    d["rezago_saber11"] = np.select([d["anios_desde_saber11"] == 0, d["anios_desde_saber11"] == 1],
                                    ["INMEDIATO", "UN_ANIO"], "DOS_O_MAS")
    canal = {canal: grupo for grupo, canales in c["canal_agrupado"].items() for canal in canales}
    d["canal_agrupado"] = d["canal"].map(canal)
    d["estrato_grupo"] = d["estrato"].map(lambda e: c["estrato_grupo"][int(e)])
    d["colegio_privado"] = (d["colegio_naturaleza"] == "PRIVADO").astype(int)
    return d


# ----------------------------------------------------------------------------- lo que se aprende del entrenamiento
def ajustar_referencias(inscritos: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, SOLO con las filas de ENTRENAMIENTO, todo lo que necesitan las columnas nuevas."""
    cfg = cfg or configuracion()
    ent = _reglas(inscritos, cfg)
    ent = ent[ent["particion"] == "ENTRENAMIENTO"]
    ref: dict = {"periodos_entrenamiento": sorted(ent["periodo"].unique()), "n_entrenamiento": len(ent)}

    # 1) Estandarización (z) con medias y desviaciones del entrenamiento
    ref["estandarizacion"] = {c: (float(ent[c].mean()), float(ent[c].std())) for c in ["punt_global", "promedio_acumulado_colegio"]}

    # 2) Segmentación de mercado con k-means: se prueba k = k_min..k_max; se elige el MENOR k cuya silueta esté a
    #    menos de tolerancia_silueta de la mejor (parsimonia: menos segmentos son más fáciles de usar para Mercadeo)
    km = cfg["caracteristicas"]["kmeans"]
    variables = km["variables"]
    medias, desv = ent[variables].mean(), ent[variables].std()
    Z = ((ent[variables] - medias) / desv).to_numpy(float)
    evaluacion = []
    for k in range(km["k_min"], km["k_max"] + 1):
        modelo = KMeans(n_clusters=k, n_init=20, random_state=km["semilla"]).fit(Z)
        evaluacion.append({"k": k, "inercia": float(modelo.inertia_),
                           "silueta": float(silhouette_score(Z, modelo.labels_, random_state=km["semilla"]))})
    tope = max(e["silueta"] for e in evaluacion) - km.get("tolerancia_silueta", 0.0)
    mejor = min((e for e in evaluacion if e["silueta"] >= tope), key=lambda e: e["k"])
    modelo = KMeans(n_clusters=mejor["k"], n_init=20, random_state=km["semilla"]).fit(Z)
    orden = np.argsort(-np.bincount(modelo.labels_))            # segmento 1 = el más grande
    centros = pd.DataFrame(modelo.cluster_centers_[orden], columns=variables)
    nombres = []
    for i, fila in centros.iterrows():
        rasgos = fila.abs().sort_values(ascending=False).index[:2]
        rasgos_txt = ", ".join(ETIQUETAS_KMEANS[v][0 if fila[v] > 0 else 1] for v in rasgos)
        nombres.append(f"S{i + 1}: " + rasgos_txt[0].upper() + rasgos_txt[1:])   # "S1: Colegio público, viven cerca…"
    ref["kmeans"] = {"modelo": modelo, "orden": orden.tolist(), "variables": variables, "medias": medias, "desv": desv,
                     "evaluacion": evaluacion, "k": mejor["k"], "nombres": nombres, "centros_z": centros}
    return ref


def construir_caracteristicas(inscritos: pd.DataFrame, ref: dict, cfg: dict | None = None) -> pd.DataFrame:
    """Devuelve la misma tabla con las columnas nuevas, aplicando lo aprendido del entrenamiento."""
    cfg = cfg or configuracion()
    d = _reglas(inscritos, cfg)
    (mg, sg), (mc, sc) = ref["estandarizacion"]["punt_global"], ref["estandarizacion"]["promedio_acumulado_colegio"]
    d["brecha_colegio_vs_icfes"] = (((d["promedio_acumulado_colegio"] - mc) / sc) - ((d["punt_global"] - mg) / sg)).round(3)
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    etiqueta = km["modelo"].predict(Z)
    posicion = {original: nuevo for nuevo, original in enumerate(km["orden"])}
    d["segmento_mercado"] = [km["nombres"][posicion[e]] for e in etiqueta]
    return d[list(inscritos.columns) + [c for c in NUEVAS if c not in inscritos.columns]]


In [ ]:
%%writefile modelado.py
"""Modelos y métricas de la matrícula (Sprint 2: regresión logística · Sprint 3: comparación con Random Forest y XGBoost).

El primer modelo del proyecto (Sprint 2) es una REGRESIÓN LOGÍSTICA sin ajuste de hiperparámetros
(config/reglas_negocio.yaml -> modelado.sprint2_logistica): es el modelo base del proyecto. En el Sprint 3 se ajustan
sus hiperparámetros con la validación y se compara con Random Forest y XGBoost.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score)


def metricas(y, p, umbral: float = 0.5) -> dict:
    """Métricas de clasificación. 'error_pct' es el lenguaje del docente: % de inscritos mal clasificados."""
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    pred = (p >= umbral).astype(int)
    unica = len(np.unique(y)) > 1
    return {
        "n": int(len(y)), "umbral": float(umbral),
        "error_pct": float(100 * (pred != y).mean()),
        "exactitud": float((pred == y).mean()),
        "auc_roc": float(roc_auc_score(y, p)) if unica else float("nan"),
        "auc_pr": float(average_precision_score(y, p)) if unica else float("nan"),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "tasa_real": float(y.mean()), "tasa_predicha_media": float(p.mean()),
    }


def tabla_metricas(resultados: dict[str, dict]) -> pd.DataFrame:
    """{nombre_modelo: metricas} -> tabla ordenada para mostrar."""
    cols = ["n", "error_pct", "exactitud", "auc_roc", "auc_pr", "brier", "precision", "recall", "f1"]
    return pd.DataFrame(resultados).T[cols].astype(float).round(4)


# =============================================================================
# Modelos de Machine Learning (regresión logística desde el Sprint 2; Random Forest y XGBoost en el Sprint 3)
# =============================================================================
# Variables de entrada. Todas se conocen ANTES de la decisión que se predice (sin fuga de datos).
# No se usa colegio_promedio_saber11 (media ICFES del colegio): Admisiones no suele tener ese dato (bitácora H-43).
VARIABLES_MATRICULA = {
    "numericas": ["punt_global", "icfes_al_cuadrado", "promedio_acumulado_colegio", "edad_al_inscribirse", "estrato",
                  "log_distancia", "dias_registro_a_inscripcion", "contactos_antes_de_inscribirse",
                  "anios_desde_saber11", "indice_vocacional", "pct_descuento_inicial",
                  "valor_neto_primer_semestre", "brecha_colegio_vs_icfes"],
    "binarias": ["entrevista_realizada", "apoyo_foes", "elegible_foes", "colegio_en_convenio", "captado_en_evento",
                 "colegio_privado", "semestre_2"],
    "categoricas": ["programa", "genero", "canal", "beca", "tipo_colegio", "regimen_salud", "segmento_mercado"],
}


def preparar_X(d: pd.DataFrame, referencias: dict, variables: dict) -> pd.DataFrame:
    """Columnas de entrada del modelo, con el tratamiento de nulos del Sprint 1 y dos derivadas simples."""
    X = d.copy()
    X["indice_vocacional"] = X["indice_vocacional"].astype("Float64").fillna(0).astype(float)   # 0 = sin entrevista
    # (entrevista_realizada distingue "sin entrevista" de un índice de 0)
    media, desv = referencias["estandarizacion"]["punt_global"]
    X["icfes_al_cuadrado"] = ((X["punt_global"] - media) / desv) ** 2       # captura la "U invertida" del ICFES
    X["semestre_2"] = X["periodo"].str.endswith("-2").astype(int)          # ingreso a mitad de año
    columnas = variables["numericas"] + variables["binarias"] + variables["categoricas"]
    X = X[columnas].copy()
    for c in variables["categoricas"]:
        X[c] = X[c].astype(str)
    for c in variables["numericas"] + variables["binarias"]:
        X[c] = X[c].astype(float)
    return X


def construir_pipeline(tipo: str, parametros: dict, variables: dict, semilla: int = 20261105):
    """Preprocesamiento + modelo en un solo objeto: se ajusta y se aplica igual en todos los conjuntos."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier

    categ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if tipo == "logistica":
        prep = ColumnTransformer([("num", StandardScaler(), variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        modelo = LogisticRegression(max_iter=5000, **parametros)
    else:
        prep = ColumnTransformer([("num", "passthrough", variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        if tipo == "random_forest":
            modelo = RandomForestClassifier(n_estimators=500, random_state=semilla, n_jobs=-1, **parametros)
        elif tipo == "xgboost":
            modelo = XGBClassifier(random_state=semilla, n_jobs=1, eval_metric="logloss", tree_method="hist",
                                   subsample=0.8, colsample_bytree=0.8, **parametros)
        else:
            raise ValueError(tipo)
    return Pipeline([("preparacion", prep), ("modelo", modelo)])


def grilla(tipo: str, cfg: dict) -> list[dict]:
    """Combinaciones de hiperparámetros a probar (config/reglas_negocio.yaml -> modelado.grillas)."""
    from itertools import product
    g = cfg["modelado"]["grillas"][tipo]
    return [dict(zip(g, valores)) for valores in product(*g.values())]


def ajustar_temporal(tipo: str, cfg: dict, Xe, ye, Xv, yv, variables: dict) -> pd.DataFrame:
    """Prueba cada combinación entrenando con ENTRENAMIENTO y midiendo en VALIDACION (2026-1)."""
    filas = []
    for params in grilla(tipo, cfg):
        modelo = construir_pipeline(tipo, params, variables, cfg["modelado"]["semilla"]).fit(Xe, ye)
        m = metricas(yv, modelo.predict_proba(Xv)[:, 1])
        filas.append({"tipo": tipo, **{f"param_{k}": v for k, v in params.items()},
                      "log_loss_validacion": m["log_loss"], "auc_validacion": m["auc_roc"],
                      "brier_validacion": m["brier"], "error_pct_validacion": m["error_pct"], "parametros": params})
    return pd.DataFrame(filas).sort_values("log_loss_validacion").reset_index(drop=True)


def intervalo_bootstrap(y, p, funcion, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% por remuestreo: se repite la medición en 1.000 muestras con reemplazo del conjunto de prueba."""
    y, p = np.asarray(y, dtype=int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(semilla)
    valores = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        valores.append(funcion(y[i], p[i], umbral))
    return float(np.percentile(valores, 2.5)), float(np.percentile(valores, 97.5))


def diferencia_error(y, p_antes, p_despues, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% de (error del modelo nuevo − error del anterior), en puntos, remuestreando los MISMOS inscritos.
    Si el intervalo incluye el 0, la diferencia puede ser azar: los dos modelos rinden igual."""
    y = np.asarray(y, dtype=int)
    falla_a = (np.asarray(p_antes) >= umbral) != y
    falla_b = (np.asarray(p_despues) >= umbral) != y
    rng = np.random.default_rng(semilla)
    dif = [100 * (falla_b[i].mean() - falla_a[i].mean()) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(dif, 2.5)), float(np.percentile(dif, 97.5))


def frase_docente(error_antes: float, error_despues: float, nombre: str, ic: tuple[float, float], n: int,
                  antes: str = "La regresión logística de la Semana 2", cuando: str = "inscritos de 2026-2") -> str:
    """La frase que pide el docente ("tenía un error del X%, este modelo del Y%"), sin afirmar una mejora que no hay."""
    num = lambda v: f"{v:.1f}".replace(".", ",")   # noqa: E731
    frase = (f"{antes} tenía un error del {num(error_antes)}%, mientras que el modelo de la Semana 3 ({nombre}) tiene "
             f"un error del {num(error_despues)}% en {n:,} {cuando}, que ningún modelo vio al entrenar.".replace(f"{n:,}", f"{n:,}".replace(",", ".")))
    dif = error_despues - error_antes
    if ic[1] < 0:
        return frase + f" Mejora {num(-dif)} puntos (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    if ic[0] > 0:
        return frase + f" Es {num(dif)} puntos peor (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    return (frase + f" La diferencia ({num(dif)} puntos) está dentro del margen del azar (IC 95%: {num(ic[0])} a "
            f"{num(ic[1])}): los dos modelos rinden igual.")


def nombres_salida(modelo) -> list[str]:
    """Nombres de las columnas que realmente ve el modelo (después del one-hot)."""
    return list(modelo.named_steps["preparacion"].get_feature_names_out())


In [ ]:
%%writefile flujo.py
"""Paso común de los sprints 3 a 6: cargar las tablas planas y construir las características del Sprint 2.

Así todos los sprints usan exactamente las mismas columnas, aprendidas solo con el entrenamiento.
"""
from __future__ import annotations

import pandas as pd

import caracteristicas as ca
import datos


def cargar_y_construir(fuente: str = "auto") -> dict:
    """Devuelve los inscritos con características, las referencias del Sprint 2 y el origen de los datos."""
    inscritos, origen = datos.cargar_inscritos(fuente)
    ref = ca.ajustar_referencias(inscritos)
    return {"inscritos": ca.construir_caracteristicas(inscritos, ref), "referencias": ref, "origen": origen}


def conjunto(d: pd.DataFrame, *nombres: str) -> pd.DataFrame:
    """Filtra por partición temporal: conjunto(d, 'ENTRENAMIENTO', 'VALIDACION')."""
    return d[d["particion"].isin(nombres)]


def modelos_finales(f: dict, parametros: dict) -> dict:
    """Reentrena los modelos finales del Sprint 3 con los hiperparámetros ya elegidos (los guarda modelos/*.json).

    Sirve en Google Colab, donde no están los archivos .joblib: con la misma semilla el resultado es idéntico.
    parametros = {"matricula": {"tipo", "parametros"}, "xgboost": {"parametros"}}
    """
    import modelado as mo
    cfg = ca.configuracion()
    semilla = cfg["modelado"]["semilla"]
    d, ref = f["inscritos"], f["referencias"]
    hist = conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
    Xh, yh = mo.preparar_X(hist, ref, mo.VARIABLES_MATRICULA), hist["matriculado"].astype(int)
    pm = parametros["matricula"]
    return {
        "referencias": ref,
        "modelo_matricula": mo.construir_pipeline(pm["tipo"], pm["parametros"], mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        "modelo_matricula_xgboost": mo.construir_pipeline("xgboost", parametros["xgboost"]["parametros"],
                                                          mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        # La regresión logística del Sprint 2 (sin ajuste de hiperparámetros), para compararla en igualdad de condiciones
        "logistica_sprint2": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"],
                                                   mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        "fondo_matricula": Xh.sample(300, random_state=1),
    }


In [ ]:
%%writefile explicabilidad.py
"""SPRINT 4 — Machine Learning explicable: ¿por qué el modelo toma sus decisiones?

SHAP (Lundberg y Lee, 2017) reparte la predicción de cada inscrito entre sus variables, como un "pago justo" en un
juego cooperativo (valores de Shapley). Para un inscrito:

    log-odds predicho = valor base (promedio) + suma de las contribuciones SHAP de sus variables

Como las contribuciones SE SUMAN, se pueden agrupar variables que miden la misma idea (por ejemplo, la beca, el
descuento y el valor neto forman "Beca y precio"). Eso hace la explicación legible para las directivas.

LIME (Ribeiro et al., 2016) es otra mirada: ajusta un modelo lineal sencillo alrededor de UN inscrito, con datos
simulados parecidos a él. Si SHAP y LIME señalan los mismos factores, la explicación es más confiable.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from caracteristicas import NOMBRE_TIPO_COLEGIO

# Variables agrupadas en conceptos de negocio (contribuciones SHAP sumadas)
GRUPOS_MATRICULA = {
    "Beca y precio": ["beca", "pct_descuento_inicial", "valor_neto_primer_semestre"],
    "Puntaje Saber 11": ["punt_global", "icfes_al_cuadrado"],
    "Distancia al campus": ["log_distancia"],
    "Canal de captación": ["canal", "captado_en_evento"],
    "Entrevista vocacional": ["entrevista_realizada", "indice_vocacional"],
    "Apoyo FOES municipal": ["apoyo_foes", "elegible_foes"],
    "Estrato y régimen de salud": ["estrato", "regimen_salud"],
    "Programa": ["programa"],
    "Ingreso a mitad de año": ["semestre_2"],
    "Colegio de procedencia": ["tipo_colegio", "colegio_privado", "colegio_en_convenio"],
    "Rendimiento en el colegio": ["promedio_acumulado_colegio", "brecha_colegio_vs_icfes"],
    "Contacto con asesores": ["contactos_antes_de_inscribirse", "dias_registro_a_inscripcion"],
    "Edad y años desde el Saber 11": ["edad_al_inscribirse", "anios_desde_saber11"],
    "Segmento de mercado": ["segmento_mercado"],
    "Género": ["genero"],
}

def _variable_de(columna: str, variables: dict) -> str:
    """Nombre original de una columna transformada (canal_REFERIDO -> canal)."""
    if columna in variables["numericas"] + variables["binarias"]:
        return columna
    return next(c for c in variables["categoricas"] if columna.startswith(c + "_"))


def contribuciones(modelo, X_fondo: pd.DataFrame, X: pd.DataFrame, variables: dict) -> tuple[pd.DataFrame, float]:
    """Valores SHAP en log-odds, ya agregados por variable ORIGINAL. Devuelve (tabla n × variables, valor base)."""
    import shap
    prep, clf = modelo.named_steps["preparacion"], modelo.named_steps["modelo"]
    nombres = list(prep.get_feature_names_out())
    T_fondo, T = prep.transform(X_fondo), prep.transform(X)
    if hasattr(clf, "coef_"):            # regresión logística: SHAP lineal (exacto)
        explicador = shap.LinearExplainer(clf, shap.maskers.Independent(T_fondo, max_samples=len(T_fondo)))
    else:                                # árboles (Random Forest, XGBoost): TreeSHAP con 100 casos de referencia
        explicador = shap.TreeExplainer(clf, data=T_fondo[:100], model_output="raw", feature_perturbation="interventional")
    exp = explicador(T)
    valores = exp.values[..., 1] if exp.values.ndim == 3 else exp.values
    base = exp.base_values[..., 1] if np.ndim(exp.base_values) == 2 else exp.base_values
    tabla = pd.DataFrame(valores, columns=nombres, index=X.index)
    tabla = tabla.T.groupby([_variable_de(c, variables) for c in nombres]).sum().T
    return tabla[[c for c in X.columns if c in tabla.columns]], float(np.mean(base))


def agrupar(contrib: pd.DataFrame, grupos: dict) -> pd.DataFrame:
    """Suma las contribuciones de las variables de cada concepto (SHAP es aditivo)."""
    return pd.DataFrame({g: contrib[[v for v in vs if v in contrib]].sum(axis=1) for g, vs in grupos.items()})


def importancia(contrib: pd.DataFrame) -> pd.Series:
    """Importancia global = promedio del valor absoluto de la contribución (cuánto mueve, en cualquier sentido)."""
    return contrib.abs().mean().sort_values(ascending=False)


def contexto(fila: pd.Series, grupo: str) -> str:
    """Dato concreto que acompaña a cada factor en la explicación en español."""
    f = fila
    textos = {
        "Beca y precio": lambda: f"beca {str(f['beca']).replace('_', ' ').title()}, descuento {float(f['pct_descuento_inicial']):.0%}",
        "Puntaje Saber 11": lambda: f"puntaje global {int(f['punt_global'])}",
        "Distancia al campus": lambda: f"vive a {float(np.exp(f['log_distancia'])):.0f} km por vía",
        "Canal de captación": lambda: f"llegó por {str(f['canal']).replace('_', ' ').lower()}",
        "Entrevista vocacional": lambda: (f"índice vocacional {float(f['indice_vocacional']):.0f} de 100" if f["entrevista_realizada"] == 1 else "sin entrevista"),
        "Apoyo FOES municipal": lambda: "con apoyo del fondo municipal" if f.get("apoyo_foes", 0) == 1 else "sin apoyo del fondo municipal",
        "Estrato y régimen de salud": lambda: f"estrato {int(f['estrato'])}",
        "Programa": lambda: f"programa {f['programa']}",
        "Ingreso a mitad de año": lambda: "ingresa en el segundo periodo del año" if f["semestre_2"] == 1 else "ingresa en el primer periodo del año",
        "Rendimiento en el colegio": lambda: f"promedio del colegio {float(f['promedio_acumulado_colegio']):.2f}",
        "Colegio de procedencia": lambda: f"colegio {NOMBRE_TIPO_COLEGIO.get(str(f['tipo_colegio']), str(f['tipo_colegio']).lower())}",
        "Contacto con asesores": lambda: f"{int(f['contactos_antes_de_inscribirse'])} contactos antes de inscribirse",
        "Edad y años desde el Saber 11": lambda: f"{int(f['edad_al_inscribirse'])} años",
        "Segmento de mercado": lambda: str(f["segmento_mercado"]),
        "Género": lambda: f"género {f['genero']}",
    }
    try:
        return textos[grupo]()
    except (KeyError, ValueError, TypeError):
        return ""


def top_factores(por_grupo: pd.DataFrame, X: pd.DataFrame, ids: pd.Series, k: int = 3, que: str = "matricularse") -> pd.DataFrame:
    """Bloque 4 del dashboard: los k conceptos que más mueven la predicción de cada inscrito, en español."""
    filas = []
    for (idx, contrib), iid in zip(por_grupo.iterrows(), ids):
        for orden, (grupo, valor) in enumerate(contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:k].items(), 1):
            sube = valor > 0
            dato = contexto(X.loc[idx], grupo)
            filas.append({"inscripcion_id": int(iid), "orden": orden, "factor": grupo,
                          "direccion": "SUBE" if sube else "BAJA", "contribucion_log_odds": round(float(valor), 4),
                          "texto": f"{grupo}{f' ({dato})' if dato else ''}: {'aumenta' if sube else 'reduce'} la probabilidad de {que}."})
    return pd.DataFrame(filas)


def explicar_lime(modelo, X_ref: pd.DataFrame, fila: pd.DataFrame, variables: dict, n_variables: int = 8,
                  semilla: int = 11) -> list[tuple[str, float]]:
    """LIME: modelo lineal local alrededor de un inscrito. Las categorías se codifican como números para LIME."""
    from lime.lime_tabular import LimeTabularExplainer
    cols = list(X_ref.columns)
    cats = [cols.index(c) for c in variables["categoricas"]]
    niveles = {cols[i]: sorted(X_ref[cols[i]].unique()) for i in cats}

    def codificar(df: pd.DataFrame) -> np.ndarray:
        A = df.copy()
        for c, nv in niveles.items():
            A[c] = A[c].map({v: k for k, v in enumerate(nv)}).fillna(0)
        return A.to_numpy(float)

    def decodificar(A: np.ndarray) -> pd.DataFrame:
        df = pd.DataFrame(A, columns=cols)
        for c, nv in niveles.items():
            df[c] = [nv[int(round(min(max(v, 0), len(nv) - 1)))] for v in df[c]]
        for c in variables["numericas"] + variables["binarias"]:
            df[c] = df[c].astype(float)
        return df

    explicador = LimeTabularExplainer(codificar(X_ref), feature_names=cols, categorical_features=cats,
                                      categorical_names={i: niveles[cols[i]] for i in cats}, class_names=["no", "sí"],
                                      discretize_continuous=True, random_state=semilla, mode="classification")
    exp = explicador.explain_instance(codificar(fila)[0], lambda A: modelo.predict_proba(decodificar(A)),
                                      num_features=n_variables, num_samples=3000)
    return exp.as_list()


def importancia_permutacion(modelo, X: pd.DataFrame, y, grupos: dict, repeticiones: int = 10,
                            semilla: int = 5) -> pd.DataFrame:
    """Chequeo independiente de SHAP: se DESORDENA un concepto (todas sus columnas a la vez) y se mide cuánto cae el
    AUC. Si cae mucho, el modelo dependía de ese concepto. No usa los valores SHAP: es otra forma de verlo."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(semilla)
    auc_base = roc_auc_score(y, modelo.predict_proba(X)[:, 1])
    filas = []
    for grupo, cols in grupos.items():
        cols = [c for c in cols if c in X]
        caidas = []
        for _ in range(repeticiones):
            Xp = X.copy()
            orden = rng.permutation(len(X))
            Xp[cols] = X[cols].to_numpy()[orden]
            caidas.append(auc_base - roc_auc_score(y, modelo.predict_proba(Xp)[:, 1]))
        filas.append({"concepto": grupo, "caida_auc": float(np.mean(caidas)), "de_caida": float(np.std(caidas))})
    return pd.DataFrame(filas).sort_values("caida_auc", ascending=False).reset_index(drop=True)


def chequeos_coherencia(X: pd.DataFrame, contrib: pd.DataFrame, por_grupo: pd.DataFrame, prob: np.ndarray,
                        y_real: pd.Series | None = None) -> pd.DataFrame:
    """¿El modelo respeta las reglas de negocio? Cada fila es una afirmación verificable con los datos."""
    from scipy.stats import spearmanr
    saber = contrib[["punt_global", "icfes_al_cuadrado"]].sum(axis=1)
    banda = pd.cut(X["punt_global"], [0, 230, 290, 330, 600], labels=["< 230", "230-289", "290-329", "≥ 330"])
    por_banda = saber.groupby(banda, observed=True).mean()
    beca = contrib["beca"].groupby(X["beca"]).mean()
    imp = importancia(por_grupo)
    filas = [
        ("Más descuento → más probabilidad de matrícula",
         spearmanr(X["pct_descuento_inicial"], contrib["pct_descuento_inicial"] + contrib["valor_neto_primer_semestre"]).statistic > 0,
         "correlación entre descuento y su contribución SHAP (con el valor neto) > 0"),
        ("Las becas pesan a favor frente a no tener beca",
         bool(beca.get("SABER_11", 0) > beca.get("NINGUNA", 0) and beca.get("MI_VOCACION", 0) > beca.get("NINGUNA", 0)),
         "contribución media de la beca: " + ", ".join(f"{k} {v:+.2f}" for k, v in beca.items())),
        ("Más distancia → menos probabilidad",
         spearmanr(X["log_distancia"], contrib["log_distancia"]).statistic < 0, "correlación entre distancia y su contribución < 0"),
        ("Saber 11 en U invertida: los puntajes muy altos tienen más opciones y se matriculan menos",
         bool(por_banda.iloc[-1] < por_banda.iloc[1:3].max()),
         "contribución media por banda: " + ", ".join(f"{k} {v:+.2f}" for k, v in por_banda.items())),
        ("La entrevista vocacional pesa a favor",
         bool(contrib["entrevista_realizada"][X["entrevista_realizada"] == 1].mean() > contrib["entrevista_realizada"][X["entrevista_realizada"] == 0].mean()),
         "contribución media con entrevista > sin entrevista"),
        ("El género NO domina las decisiones",
         bool(list(imp.index).index("Género") >= len(imp) // 2 and imp["Género"] < 0.15 * imp.iloc[0]),
         f"puesto {list(imp.index).index('Género') + 1} de {len(imp)}; importancia {imp['Género']:.3f} vs. {imp.iloc[0]:.3f} del primero"),
    ]
    if y_real is not None:
        tasas = pd.DataFrame({"predicha": pd.Series(prob, index=X.index), "real": y_real.astype(float)}).groupby(X["genero"]).mean()
        brecha_pred = abs(tasas.loc["F", "predicha"] - tasas.loc["M", "predicha"])
        brecha_real = abs(tasas.loc["F", "real"] - tasas.loc["M", "real"])
        filas.append(("Paridad por género: el modelo no crea una brecha que no exista en los datos",
                      bool(brecha_pred <= brecha_real + 0.03),
                      f"brecha predicha F-M {brecha_pred:.3f} vs. brecha real {brecha_real:.3f}"))
    return pd.DataFrame(filas, columns=["regla", "cumple", "evidencia"])


def comparar_con_generador(X: pd.DataFrame, contrib: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Los datos son sintéticos: se conocen los efectos VERDADEROS (config: generador.matricula).
    ¿La explicación SHAP recupera la dirección de cada efecto real? Las variables sin efecto real deberían pesar poco."""
    from scipy.stats import spearmanr
    b = cfg["generador"]["matricula"]
    canal_shap = (contrib["canal"] + contrib["captado_en_evento"]).groupby(X["canal"]).mean()
    canal_real = pd.Series(b["beta_canal"]).reindex(canal_shap.index)
    estr_shap = contrib["estrato"].groupby(X["estrato"]).mean()
    estr_real = pd.Series({int(k): v for k, v in b["beta_estrato"].items()}).reindex(estr_shap.index)
    sentido = lambda v: "positivo" if v > 0 else "negativo"   # noqa: E731
    filas = [
        ("Descuento (beca)", f"positivo (β = {b['beta_descuento']})",
         sentido(spearmanr(X["pct_descuento_inicial"], contrib["pct_descuento_inicial"] + contrib["valor_neto_primer_semestre"] + contrib["beca"]).statistic)),
        ("Distancia al campus", f"negativo (β = {b['beta_log_distancia']})", sentido(spearmanr(X["log_distancia"], contrib["log_distancia"]).statistic)),
        ("Saber 11 (término cuadrático)", f"U invertida (β = {b['beta_icfes_cuadratico']})",
         "U invertida" if contrib["icfes_al_cuadrado"].corr(X["icfes_al_cuadrado"]) < 0 else "no recupera la U"),
        ("Entrevista vocacional", f"positivo (β = {b['beta_entrevista']})",
         sentido(contrib["entrevista_realizada"].corr(X["entrevista_realizada"]))),
        ("Apoyo FOES", f"positivo (β = {b['beta_foes']})", sentido(contrib["apoyo_foes"].corr(X["apoyo_foes"]))),
        ("Programa SSIA", f"positivo (β = {b['beta_programa_ssia']})",
         sentido(contrib["programa"][X["programa"] == "SSIA"].mean() - contrib["programa"][X["programa"] == "AD"].mean())),
        ("Ingreso a mitad de año", f"negativo (β = {b['beta_periodo_2']})", sentido(contrib["semestre_2"].corr(X["semestre_2"]))),
        ("Canal (orden de los 8 canales)", "inmersión > referido > campus day/convenio > web > redes > llamada > expo",
         f"correlación de Spearman con el orden real = {spearmanr(canal_shap, canal_real).statistic:.2f}"),
        ("Estrato (orden de los 6 estratos)", "estrato 1 el más bajo; 4 el más alto",
         f"correlación de Spearman con el orden real = {spearmanr(estr_shap, estr_real).statistic:.2f}"),
        ("Género (sin efecto real)", "ninguno (β = 0)", f"importancia {contrib['genero'].abs().mean():.3f} log-odds"),
        ("Promedio del colegio (sin efecto real en la matrícula)", "ninguno (β = 0)",
         f"importancia {contrib['promedio_acumulado_colegio'].abs().mean():.3f} log-odds"),
    ]
    tabla = pd.DataFrame(filas, columns=["efecto", "verdadero (generador)", "según SHAP"])
    tabla["coincide"] = [
        (r["según SHAP"] in r["verdadero (generador)"]) if r["según SHAP"] in ("positivo", "negativo", "U invertida")
        else (float(r["según SHAP"].split("= ")[-1]) > 0.5 if "Spearman" in r["según SHAP"]
              else float(r["según SHAP"].split()[1]) < 0.05)
        for _, r in tabla.iterrows()]
    return tabla


In [ ]:
import sys
import warnings

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
EN_COLAB = True

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, GRIS = "#1B365D", "#E07B39", "#9AA5B1"

In [ ]:
import caracteristicas as ca
import flujo
import modelado as mo

# Lee las tablas planas por la API de Supabase (solo lectura): dataset_matricula y la rúbrica de la entrevista
# (dataset_entrevista). Después construye las columnas del Sprint 2.
f = flujo.cargar_y_construir("supabase" if EN_COLAB else "auto")
d, ref, cfg = f["inscritos"], f["referencias"], ca.configuracion()
print("Origen de los datos:", f["origen"])
print(f"Inscritos: {len(d):,} · con entrevista: {int(d['entrevista_realizada'].sum()):,}")
print(d["particion"].value_counts().to_string())

In [ ]:
# Hiperparámetros elegidos en el Sprint 3 (sprints/sprint_3_modelado/modelos/*.json)
PARAMETROS_FINALES = {'matricula': {'parametros': {'C': 0.01}, 'tipo': 'logistica'},
 'xgboost': {'parametros': {'learning_rate': 0.03,
                            'max_depth': 2,
                            'min_child_weight': 5,
                            'n_estimators': 200}}}

art = flujo.modelos_finales(f, PARAMETROS_FINALES)     # mismo resultado que los .joblib del repositorio
print("Modelos reentrenados:", ", ".join(k for k in art if k.startswith(("modelo", "logistica"))))

## 1. Qué es SHAP
SHAP reparte la predicción de cada inscrito entre sus variables, como un **reparto justo** entre jugadores de un equipo
(valores de Shapley, teoría de juegos). Para cada inscrito:

`log-odds predicho = valor base (inscrito promedio) + suma de las contribuciones de sus variables`

- Una contribución **positiva** sube la probabilidad de matrícula y una **negativa** la baja.
- Como las contribuciones **se suman**, las variables que miden la misma idea se agrupan en conceptos. Por ejemplo,
  la beca, el descuento y el valor neto forman "Beca y precio".
- En la regresión logística, SHAP es exacto: contribución = coeficiente × (valor − promedio).

In [ ]:
import explicabilidad as ex
V = mo.VARIABLES_MATRICULA
hist = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
X_hist = mo.preparar_X(hist, ref, V)
lm, xgb, fondo = art["modelo_matricula"], art["modelo_matricula_xgboost"], art["fondo_matricula"]
c_lm, base_lm = ex.contribuciones(lm, fondo, X_hist, V)
c_xg, _ = ex.contribuciones(xgb, fondo, X_hist, V)
g_lm, g_xg = ex.agrupar(c_lm, ex.GRUPOS_MATRICULA), ex.agrupar(c_xg, ex.GRUPOS_MATRICULA)
print(f"Valor base (probabilidad de un inscrito promedio): {1 / (1 + np.exp(-base_lm)):.1%}")
# Comprobación de la aditividad: base + suma de contribuciones = predicción del modelo
pred = lm.predict_proba(X_hist)[:, 1]
reconstruida = 1 / (1 + np.exp(-(base_lm + c_lm.sum(axis=1))))
print("Diferencia máxima entre la predicción y base + suma de SHAP:", float(np.abs(pred - reconstruida).max()).__round__(6))
display(pd.DataFrame([(g, ", ".join(v)) for g, v in ex.GRUPOS_MATRICULA.items()], columns=["concepto", "variables"]))

## 2. Importancia global
Importancia = el promedio del **valor absoluto** de la contribución, es decir, cuánto mueve un concepto la predicción
en cualquier sentido. Se compara con XGBoost: si dos modelos distintos señalan lo mismo, la conclusión es más sólida.

In [ ]:
imp = pd.DataFrame({"regresion_logistica": ex.importancia(g_lm), "xgboost": ex.importancia(g_xg)}).sort_values("regresion_logistica", ascending=False)
imp["% del total"] = 100 * imp["regresion_logistica"] / imp["regresion_logistica"].sum()
display(imp.round(3))
fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, col, color in [(ejes[0], "regresion_logistica", AZUL), (ejes[1], "xgboost", NARANJA)]:
    s = imp[col].sort_values(); ax.barh(s.index, s.values, color=color); ax.set_title(f"Importancia SHAP · {col}")
plt.tight_layout(); plt.show()
veces = imp.loc["Puntaje Saber 11", "regresion_logistica"] / imp.loc["Edad y años desde el Saber 11", "regresion_logistica"]
print(f"El modelo le da {veces:.1f} veces más peso al puntaje Saber 11 que a la edad.")

**En español sencillo.** Lo que más decide si un inscrito se matricula es **cuánto le cuesta** (beca y descuento). Le
siguen la **entrevista vocacional**, su **puntaje Saber 11**, la **distancia** desde su casa al campus y el **canal** por
el que llegó. La edad y el género pesan poco. Tiene sentido para la Universidad: el precio y la cercanía son barreras
reales, y quien tiene interés vocacional claro se compromete más.

## 3. Cómo cambia cada factor

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(19, 4))
ejes[0].scatter(X_hist["pct_descuento_inicial"], c_lm[["pct_descuento_inicial", "valor_neto_primer_semestre", "beca"]].sum(axis=1), s=6, alpha=0.4, color=AZUL)
ejes[0].set(title="Beca y descuento", xlabel="descuento 1er semestre", ylabel="contribución SHAP")
ejes[1].scatter(np.exp(X_hist["log_distancia"]), c_lm["log_distancia"], s=6, alpha=0.4, color=AZUL); ejes[1].set(title="Distancia (km)")
ejes[2].scatter(X_hist["punt_global"], c_lm[["punt_global", "icfes_al_cuadrado"]].sum(axis=1), s=6, alpha=0.4, color=AZUL)
ejes[2].axvline(310, color=NARANJA, ls="--"); ejes[2].set(title="Saber 11: U invertida", xlabel="puntaje global")
canal = (c_lm["canal"] + c_lm["captado_en_evento"]).groupby(X_hist["canal"]).mean().sort_values()
ejes[3].barh(canal.index, canal.values, color=[NARANJA if x < 0 else AZUL for x in canal.values]); ejes[3].set(title="Canal (contribución media)")
plt.tight_layout(); plt.show()

## 4. Tres inscritos de la campaña 2027-1: SHAP vs. LIME
LIME explica **un** inscrito ajustando un modelo lineal sencillo con miles de "vecinos" simulados parecidos a él. Si
SHAP y LIME señalan los mismos factores, la explicación es más confiable.

In [ ]:
camp = flujo.conjunto(d, "CAMPANA")
X_camp = mo.preparar_X(camp, ref, V)
prob_camp = pd.Series(lm.predict_proba(X_camp)[:, 1], index=camp.index)
casos = {n: (prob_camp - prob_camp.quantile(q)).abs().idxmin() for n, q in [("ALTA", 0.9), ("MEDIA", 0.5), ("BAJA", 0.1)]}
c_camp, _ = ex.contribuciones(lm, fondo, X_camp.loc[list(casos.values())], V)
g_camp = ex.agrupar(c_camp, ex.GRUPOS_MATRICULA)
for nombre, idx in casos.items():
    print(f"\n=== Propensión {nombre} · inscrito {int(camp.loc[idx, 'inscripcion_id'])} · probabilidad {prob_camp[idx]:.0%}")
    top = g_camp.loc[idx].reindex(g_camp.loc[idx].abs().sort_values(ascending=False).index)[:4]
    print("SHAP (conceptos):", "; ".join(f"{k} {v:+.2f}" for k, v in top.items()))
    lime = ex.explicar_lime(lm, X_hist.sample(800, random_state=2), X_camp.loc[[idx]], V, n_variables=5)
    print("LIME (reglas locales):", "; ".join(f"{k} {w:+.3f}" for k, w in lime))

## 5. Chequeo independiente: importancia por permutación
Se **desordena** un concepto en la prueba 2026-2, dejando lo demás igual, y se mide cuánto cae el AUC del modelo. No
usa SHAP: es otra forma de preguntar de qué depende el modelo.

In [ ]:
ev, t = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION"), flujo.conjunto(d, "PRUEBA")
pm = PARAMETROS_FINALES["matricula"]     # el modelo elegido, reentrenado sin la prueba para medir en ella
lm_ev = mo.construir_pipeline(pm["tipo"], pm["parametros"], V, cfg["modelado"]["semilla"]).fit(
    mo.preparar_X(ev, ref, V), ev["matriculado"].astype(int))
perm = ex.importancia_permutacion(lm_ev, mo.preparar_X(t, ref, V), t["matriculado"].astype(int), ex.GRUPOS_MATRICULA)
display(perm.head(8).round(4))

## 6. Coherencia con las reglas de negocio y equidad
Cada afirmación se comprueba con los datos. La equidad se revisa de dos formas: el género debe pesar poco, y el
modelo no debe crear una diferencia entre hombres y mujeres que no exista en los datos reales.

In [ ]:
coh = ex.chequeos_coherencia(X_hist, c_lm, g_lm, lm.predict_proba(X_hist)[:, 1], hist["matriculado"])
display(coh)
print(f"Cumple {int(coh['cumple'].sum())} de {len(coh)} reglas")

## 7. ¿El modelo encontró las causas reales?
Como los datos son sintéticos, se conocen los efectos **verdaderos** con los que se generó la matrícula
(`config/reglas_negocio.yaml → generador.matricula`). Es una prueba que con datos reales no se puede hacer: ¿la
explicación recupera la dirección de cada causa real?

In [ ]:
gen = ex.comparar_con_generador(X_hist, c_lm, cfg)
display(gen)
print(f"Coinciden {int(gen['coincide'].sum())} de {len(gen)} efectos.")

**Lectura.** Los efectos grandes (beca, distancia, U invertida del Saber 11, entrevista, FOES, canal) se recuperan bien.
Fallan el orden del estrato (pocos casos en estratos 5 y 6) y el promedio del colegio: no tiene efecto en el generador,
pero el modelo le da algo de peso porque en los datos va de la mano con otras variables. **Lección:** una explicación
muestra lo que el modelo aprendió, no garantiza la causa.

## 8. Explicaciones para el tablero
Para cada inscrito se generan los **3 factores** que más mueven su probabilidad de matrícula, en español. Esta es la
tabla `explicacion_prediccion` (bloque 4 del tablero).

In [ ]:
X_todos = mo.preparar_X(d, ref, V)
c_todos, _ = ex.contribuciones(lm, fondo, X_todos, V)
explicacion = ex.top_factores(ex.agrupar(c_todos, ex.GRUPOS_MATRICULA), X_todos, d["inscripcion_id"])
print(f"\nTabla del bloque 4: {len(explicacion):,} filas (3 por inscrito). Ejemplo:")
display(explicacion.head(6))

## Conclusiones del Sprint 4
1. **Por qué decide así:** la beca y el precio explican más de una cuarta parte de las decisiones. Les siguen la
   entrevista, el Saber 11 (en U invertida), la distancia y el canal. La edad y el género pesan poco.
2. **Coherencia:** el modelo respeta las reglas de negocio y no crea brechas por género.
3. **Rigor:** SHAP recupera la mayoría de las causas reales del generador, y la importancia por permutación lo
   confirma.
4. **Para el tablero:** cada inscrito tiene sus 3 razones principales en español (`explicacion_prediccion`).